In [1]:
"""
Complete simulation suite for:
  A Closed-Form Rate-Variance Pareto Frontier for Stochastic Momentum SGD

Fixes applied:
  - stable_set_nesterov: now correctly returns None when s >= 2 (no stable beta).
  - fig7: handles Nesterov-unstable regime; uses s = 1.5 and s = 1.8.
  - fig6: switch point now printed and used in text (0.468, not 0.21).
  - Added numeric verification of variance formulas against Lyapunov solve.

Generates: fig1_det.png, fig2_stoch.png, fig3_hb_rho.png,
           fig4_pareto.png, fig5_logistic.png, fig6_multidim.png,
           fig7_nesterov.png, table_logistic.csv
"""

import csv
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams.update({"font.size": 11, "figure.dpi": 120})

rng = np.random.default_rng(42)

# ----------------------------------------------------------------------
# Core formulas
# ----------------------------------------------------------------------

def v_gd(eta, lam, sigma2):
    """1D GD limiting variance."""
    return eta * sigma2 / (lam * (2.0 - eta * lam))

def v_hb(eta, lam, sigma2, beta):
    """1D Heavy-ball limiting variance."""
    s = eta * lam
    return (eta * sigma2 * (1.0 + beta)
            / ((1.0 - beta) * lam * (2.0 * (1.0 + beta) - s)))

def v_nesterov(eta, lam, sigma2, beta):
    """1D Nesterov limiting variance (corrected formula)."""
    s = eta * lam
    num = eta * sigma2 * (1.0 + beta * (1.0 - s))
    den = lam * (1.0 - beta * (1.0 - s)) * (2.0 * (1.0 + beta) - s * (1.0 + 2.0 * beta))
    return num / den

def rho_s(s, beta):
    """Modal spectral radius for Heavy-ball at normalized step s."""
    a = 1.0 + beta - s
    disc = a * a - 4.0 * beta
    if disc >= 0:
        r1 = 0.5 * (a + np.sqrt(disc))
        r2 = 0.5 * (a - np.sqrt(disc))
        return max(abs(r1), abs(r2))
    return np.sqrt(beta)

def rho_s_nesterov(s, beta):
    """Modal spectral radius for Nesterov at normalized step s."""
    a = (1.0 + beta) * (1.0 - s)
    b = beta * (1.0 - s)
    disc = a * a - 4.0 * b
    if disc >= 0:
        r1 = 0.5 * (a + np.sqrt(disc))
        r2 = 0.5 * (a - np.sqrt(disc))
        return max(abs(r1), abs(r2))
    return np.sqrt(abs(b))

def vbar_s(s, beta):
    return s * (1.0 + beta) / ((1.0 - beta) * (2.0 * (1.0 + beta) - s))

def vbar_nesterov(s, beta):
    num = s * (1.0 + beta * (1.0 - s))
    den = (1.0 - beta * (1.0 - s)) * (2.0 * (1.0 + beta) - s * (1.0 + 2.0 * beta))
    return num / den

def stable_set(s):
    """Heavy-ball stable beta lower bound for given s."""
    return max(0.0, s / 2.0 - 1.0)

def stable_set_nesterov(s):
    """
    Nesterov stable beta interval for given s.
    Stability: s < 1 + 1/(1+2 beta).
      - s <= 1: all beta in [0,1) are stable. Return (0.0, 1.0).
      - 1 < s < 2: beta < (2-s)/(2(s-1)). Return (0.0, beta_max).
      - s >= 2: no beta >= 0 is stable. Return None.
    """
    if s <= 1.0:
        return (0.0, 1.0)
    if s >= 2.0:
        return None
    beta_max = (2.0 - s) / (2.0 * (s - 1.0))
    return (0.0, min(beta_max, 1.0))

# ----------------------------------------------------------------------
# Lyapunov cross-check for variance formulas
# ----------------------------------------------------------------------
def lyap_solve_2d(M, B, sigma2):
    """Solve S = M S M^T + sigma^2 B B^T for 2x2 M via vectorization."""
    I = np.eye(4)
    K = np.kron(M, M)
    b = sigma2 * np.outer(B.flatten(), B.flatten()).flatten()
    vecS = np.linalg.solve(I - K, b)
    return vecS.reshape(2, 2)

def verify_formulas():
    print("\n=== Formula verification against Lyapunov solve ===")
    tests = [
        ("HB",  0.5, 0.3, 1.0, 0.5),
        ("HB",  1.5, 0.4, 1.5, 1.0),   # lam=1.5, eta=1.0 → s=1.5
        ("Nest", 0.5, 0.3, 1.0, 0.5),
        ("Nest", 0.8, 0.5, 1.0, 0.8/1.0),
    ]
    for kind, s, beta, lam, eta in tests:
        sigma2 = 1.0
        if kind == "HB":
            A = 1.0 + beta - s
            B0 = -beta
            M = np.array([[A, B0], [1.0, 0.0]])
            Bvec = np.array([-eta, 0.0])
            S = lyap_solve_2d(M, Bvec, sigma2)
            v_num = S[0, 0]
            v_formula = v_hb(eta, lam, sigma2, beta)
        else:
            A = (1.0 + beta) * (1.0 - s)
            B0 = -beta * (1.0 - s)
            M = np.array([[A, B0], [1.0, 0.0]])
            Bvec = np.array([-eta, 0.0])
            S = lyap_solve_2d(M, Bvec, sigma2)
            v_num = S[0, 0]
            v_formula = v_nesterov(eta, lam, sigma2, beta)
        rel = abs(v_num - v_formula) / max(abs(v_num), 1e-12)
        print(f"  {kind:4s} s={s:.2f} beta={beta:.2f}: "
              f"Lyapunov={v_num:.6f}, formula={v_formula:.6f}, rel_err={rel:.2e}")

# ----------------------------------------------------------------------
# Figure 1: deterministic GD stability boundary
# ----------------------------------------------------------------------
def fig1():
    lam = 10.0
    etas = [(0.15, "η=0.15 (stable)", "-"),
            (0.20, "η=0.20 (boundary)", "--"),
            (0.25, "η=0.25 (unstable)", ":")]
    K = 30
    fig, ax = plt.subplots(figsize=(8, 5))
    for eta, label, style in etas:
        x = 1.0
        xs = [x]
        for _ in range(K):
            x = x - eta * lam * x
            xs.append(abs(x))
        ax.semilogy(range(K + 1), xs, style, label=label, linewidth=2)
    ax.set_xlabel("Iteration $k$")
    ax.set_ylabel("$|x_k|$ (log scale)")
    ax.set_title("Deterministic GD on $f(x)=5x^2$ ($\\lambda=10$)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("fig1_det.png")
    plt.close()
    print("Saved fig1_det.png")

# ----------------------------------------------------------------------
# Figure 2: SGD limiting variance
# ----------------------------------------------------------------------
def simulate_sgd_1d(eta, lam, sigma2, N=5000, K=2000, burnin=1500, seed=0):
    rng_local = np.random.default_rng(seed)
    x = np.ones((N, K + 1))
    for k in range(K):
        x[:, k + 1] = x[:, k] - eta * lam * x[:, k] - eta * rng_local.normal(
            0.0, np.sqrt(sigma2), N)
    return x[:, burnin:].flatten()

def fig2():
    lam = 10.0
    sigma2 = 1.0
    etas = np.array([0.02, 0.05, 0.08, 0.10, 0.12, 0.15, 0.18])
    v_theory = np.array([v_gd(e, lam, sigma2) for e in etas])

    N = 5000
    v_mc = np.zeros_like(etas)
    v_std = np.zeros_like(etas)
    for i, e in enumerate(etas):
        samples = simulate_sgd_1d(e, lam, sigma2, N=N, seed=100 + i)
        v_mc[i] = samples.var()
        n_batches = 50
        batches = np.array_split(samples, n_batches)
        batch_vars = np.array([b.var() for b in batches])
        v_std[i] = batch_vars.std(ddof=1) / np.sqrt(n_batches)

    rel_err = np.abs(v_mc - v_theory) / v_theory
    print(f"[Fig2] Max relative discrepancy: {rel_err.max()*100:.3f}%")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(etas, v_theory, "-", color="tab:blue", label="Analytic", linewidth=2)
    ax.errorbar(etas, v_mc, yerr=2 * v_std, fmt="o", color="tab:red",
                label="Monte Carlo (95% CI, batch means)", capsize=5)
    ax.set_xlabel("$\\eta$")
    ax.set_ylabel("Limiting variance $v$")
    ax.set_title("SGD limiting variance: analytic vs. Monte Carlo")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("fig2_stoch.png")
    plt.close()

# ----------------------------------------------------------------------
# Figure 3: Heavy-ball spectral radius with stability boundary
# ----------------------------------------------------------------------
def fig3():
    eta_vals = np.linspace(0.005, 0.30, 200)
    beta_vals = np.linspace(0.0, 0.95, 200)
    lam_max = 20.0
    lam_min = 2.0

    E, B = np.meshgrid(eta_vals, beta_vals)
    RHO = np.zeros_like(E)
    for i in range(len(beta_vals)):
        for j in range(len(eta_vals)):
            s1 = eta_vals[j] * lam_min
            s2 = eta_vals[j] * lam_max
            RHO[i, j] = max(rho_s(s1, beta_vals[i]), rho_s(s2, beta_vals[i]))

    fig, ax = plt.subplots(figsize=(8, 5))
    cs = ax.contourf(E, B, RHO, levels=25, cmap="viridis")
    plt.colorbar(cs, ax=ax, label="$\\rho(M)$")
    eta_bd = 2.0 * (1.0 + beta_vals) / lam_max
    mask = eta_bd <= eta_vals[-1]
    ax.plot(eta_bd[mask], beta_vals[mask], "r--", linewidth=2,
            label="$\\eta=2(1+\\beta)/\\lambda_{\\max}$")
    ax.contour(E, B, RHO, levels=[1.0], colors="white", linewidths=2)
    ax.set_xlabel("$\\eta$")
    ax.set_ylabel("$\\beta$")
    ax.set_title("Heavy-ball worst modal spectral radius")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig("fig3_hb_rho.png")
    plt.close()
    print("Saved fig3_hb_rho.png")

# ----------------------------------------------------------------------
# Figure 4: Pareto geometry (Heavy-ball)
# ----------------------------------------------------------------------
def fig4():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, s in zip(axes, [0.5, 2.0]):
        betas = np.linspace(0.0, 0.99, 800)
        rhos = np.array([rho_s(s, b) for b in betas])
        vbars = np.array([vbar_s(s, b) for b in betas])

        beta_min = stable_set(s)
        mask_stable = betas > beta_min

        beta_rate = (1.0 - np.sqrt(s)) ** 2
        if s < 1.0:
            mask_pareto = betas <= beta_rate
        elif np.isclose(s, 1.0):
            mask_pareto = np.abs(betas) < 1e-6
        else:
            beta_var = np.sqrt(s) - 1.0
            mask_pareto = (betas >= beta_rate) & (betas <= beta_var)

        mask_all = mask_stable & mask_pareto

        ax.plot(rhos[mask_stable], vbars[mask_stable], "-",
                color="tab:blue", alpha=0.4, linewidth=1.5,
                label="Stable (dominated)")
        ax.plot(rhos[mask_all], vbars[mask_all], "-",
                color="tab:red", linewidth=3.5, label="Pareto-optimal")

        if s < 1.0:
            ax.plot(rho_s(s, 0.0), vbar_s(s, 0.0), "ko", ms=8, label="$\\beta=0$")
            ax.plot(rho_s(s, beta_rate), vbar_s(s, beta_rate), "k^", ms=8,
                    label="$\\beta_{\\mathrm{rate}}$")
        else:
            ax.plot(rho_s(s, beta_rate), vbar_s(s, beta_rate), "k^", ms=8,
                    label="$\\beta_{\\mathrm{rate}}$")
            beta_var = np.sqrt(s) - 1.0
            ax.plot(rho_s(s, beta_var), vbar_s(s, beta_var), "ks", ms=8,
                    label="$\\beta_{\\mathrm{var}}$")

        ax.set_xlabel("$\\rho_s(\\beta)$")
        ax.set_ylabel("$\\bar v_s(\\beta)$")
        ax.set_title(f"$s=\\eta\\lambda={s}$")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("fig4_pareto.png")
    plt.close()
    print("Saved fig4_pareto.png")

# ----------------------------------------------------------------------
# Figure 6: Multidimensional kink example (FIXED switch point)
# ----------------------------------------------------------------------
def fig6():
    lam1, lam2 = 1.0, 10.0
    sig1, sig2 = 1.0, 1.0
    eta = 0.1
    s1, s2 = eta * lam1, eta * lam2

    betas = np.linspace(0.0, 0.99, 800)
    rho1 = np.array([rho_s(s1, b) for b in betas])
    rho2 = np.array([rho_s(s2, b) for b in betas])
    rho = np.maximum(rho1, rho2)
    V = np.array([(sig1 ** 2 / lam1 ** 2) * vbar_s(s1, b)
                  + (sig2 ** 2 / lam2 ** 2) * vbar_s(s2, b) for b in betas])

    # Exact switch point: mode 1 enters complex-root regime at beta_rate = (1-sqrt(s1))^2
    beta_switch_exact = (1.0 - np.sqrt(s1)) ** 2
    print(f"[Fig6] Exact switch beta* = (1-sqrt({s1}))^2 = {beta_switch_exact:.4f}")

    # Pareto front
    order = np.argsort(rho)
    rho_sorted, V_sorted = rho[order], V[order]
    pareto_mask = np.zeros_like(rho_sorted, dtype=bool)
    v_min_so_far = np.inf
    for i in range(len(rho_sorted) - 1, -1, -1):
        if V_sorted[i] < v_min_so_far:
            pareto_mask[i] = True
            v_min_so_far = V_sorted[i]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(betas, rho1, "--", label=f"mode 1 ($s_1={s1}$)")
    axes[0].plot(betas, rho2, "--", label=f"mode 2 ($s_2={s2}$)")
    axes[0].plot(betas, rho, "-", color="k", linewidth=2, label="$\\rho(\\beta)=\\max_i$")
    axes[0].axvline(beta_switch_exact, color="r", linestyle=":",
                    label=f"switch $\\beta^\\star\\approx{beta_switch_exact:.3f}$")
    axes[0].set_xlabel("$\\beta$")
    axes[0].set_ylabel("Modal spectral radius")
    axes[0].set_title("Dominant-mode switching")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(rho, V, "-", color="tab:blue", alpha=0.4, linewidth=1.5)
    axes[1].plot(rho_sorted[pareto_mask], V_sorted[pareto_mask], "-",
                 color="tab:red", linewidth=3, label="Pareto front")
    axes[1].plot(rho_s(s1, beta_switch_exact),
                 (sig1 ** 2 / lam1 ** 2) * vbar_s(s1, beta_switch_exact)
                 + (sig2 ** 2 / lam2 ** 2) * vbar_s(s2, beta_switch_exact),
                 "k*", ms=14, label="kink")
    axes[1].set_xlabel("$\\rho(\\beta)$")
    axes[1].set_ylabel("$V(\\beta)$")
    axes[1].set_title("Multidimensional Pareto front with kink")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("fig6_multidim.png")
    plt.close()

# ----------------------------------------------------------------------
# Figure 7: Nesterov vs Heavy-ball Pareto comparison (FIXED)
# ----------------------------------------------------------------------
def fig7():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, s in zip(axes, [1.5, 1.8]):
        betas = np.linspace(0.0, 0.95, 800)

        # Heavy-ball
        rho_hb = np.array([rho_s(s, b) for b in betas])
        v_hb_arr = np.array([vbar_s(s, b) for b in betas])
        beta_min_hb = stable_set(s)
        mask_hb = betas > beta_min_hb

        # Nesterov
        stab = stable_set_nesterov(s)
        if stab is None:
            ax.text(0.5, 0.5, f"Nesterov unstable for all $\\beta\\ge0$ at $s={s}$",
                    ha="center", va="center", transform=ax.transAxes, fontsize=12)
            ax.plot(rho_hb[mask_hb], v_hb_arr[mask_hb], "-", color="tab:blue",
                    linewidth=2, label="Heavy-ball")
        else:
            beta_min_nes, beta_max_nes = stab
            print(f"[Fig7] s={s}: Nesterov stable for beta in "
                  f"({beta_min_nes:.3f}, {beta_max_nes:.3f})")
            mask_nes = (betas > beta_min_nes) & (betas < beta_max_nes)
            rho_nes = np.array([rho_s_nesterov(s, b) for b in betas])
            v_nes_arr = np.array([vbar_nesterov(s, b) for b in betas])
            valid = mask_nes & np.isfinite(v_nes_arr) & (v_nes_arr > 0)

            ax.plot(rho_hb[mask_hb], v_hb_arr[mask_hb], "-", color="tab:blue",
                    linewidth=2, label="Heavy-ball")
            ax.plot(rho_nes[valid], v_nes_arr[valid], "-", color="tab:orange",
                    linewidth=2, label="Nesterov")

        ax.set_xlabel("$\\rho_s(\\beta)$")
        ax.set_ylabel("$\\bar v_s(\\beta)$")
        ax.set_title(f"$s={s}$")
        ax.legend()
        ax.grid(True, alpha=0.3)
        # Limit axes to reasonable ranges
        ax.set_xlim(0.0, 1.05)
        ax.set_ylim(0.0, 15.0)

    plt.tight_layout()
    plt.savefig("fig7_nesterov.png")
    plt.close()
    print("Saved fig7_nesterov.png")

# ----------------------------------------------------------------------
# Figure 5 + table: logistic regression
# ----------------------------------------------------------------------
def estimate_mode_noise(grads, eigvecs):
    """
    Estimate per-mode noise variances sigma_i^2 by projecting the empirical
    gradient covariance onto the Hessian eigenbasis.

    Parameters
    ----------
    grads : np.ndarray, shape (n_samples, d)
        Mini-batch gradients sampled at w*.
    eigvecs : np.ndarray, shape (d, d)
        Columns are Hessian eigenvectors (from torch.linalg.eigh).

    Returns
    -------
    sigma2_modes : np.ndarray, shape (d,)
        Per-mode noise variances sigma_i^2.
    """
    Sigma_hat = np.cov(grads.T)                       # d x d
    sigma2_modes = np.einsum('ij,jk,ik->i',
                             eigvecs.T, Sigma_hat, eigvecs.T)
    return np.maximum(sigma2_modes, 0.0)              # guard numerical noise


def compute_V_theory(eigvals, sigma2_modes, eta, beta):
    """
    Scalar-mode prediction of the steady-state variance:
        V_theory = sum_i (sigma_i^2 / lambda_i^2) * vbar_s(s_i, beta)
    using only stable modes s_i = eta * lambda_i < 2(1+beta).
    """
    V = 0.0
    for lam, sig2 in zip(eigvals, sigma2_modes):
        s_i = eta * lam
        if s_i >= 2.0 * (1.0 + beta) - 1e-12:         # unstable mode -> skip
            continue
        V += (sig2 / lam**2) * vbar_s(s_i, beta)
    return float(V)


def logistic_experiment():
    import torch

    torch.set_default_dtype(torch.float64)   # KEY FIX #1: float64 precision
    torch.manual_seed(42)

    N, d = 2000, 20
    gamma = 0.1
    w_true = torch.randn(d) * 0.5
    X = torch.randn(N, d)
    y = torch.sign(X @ w_true)
    y[y == 0] = 1.0

    # --- Find w* by full-batch GD (float64) ---
    w_star = torch.zeros(d)
    for _ in range(20000):
        z = -y * (X @ w_star)
        grad = (X.T @ (-y * torch.sigmoid(z))) / N + gamma * w_star
        w_star = w_star - 0.5 * grad

    p = torch.sigmoid(X @ w_star)
    W = p * (1 - p)
    H = (X.T @ (W.unsqueeze(1) * X)) / N + gamma * torch.eye(d)
    eigvals_t, eigvecs_t = torch.linalg.eigh(H)
    eigvals = eigvals_t.numpy()
    eigvecs = eigvecs_t.numpy()
    lam_max = eigvals.max()
    lam_min = eigvals.min()
    print(f"[LogReg] lam_max={lam_max:.4f}, lam_min={lam_min:.4f}, "
          f"kappa={lam_max/lam_min:.3f}")

    # --- Estimate local noise covariance at w* in the Hessian eigenbasis ---
    B = 50
    rng_noise = np.random.default_rng(2024)
    B_est = 200
    n_est = 2000
    grads_at_wstar = np.zeros((n_est, d))
    for k in range(n_est):
        idx_b = rng_noise.integers(0, N, B_est)
        Xb = X[idx_b]
        yb = y[idx_b]
        z = -yb * (Xb @ w_star)
        g = (Xb.T @ (-yb * torch.sigmoid(z))) / B_est + gamma * w_star
        grads_at_wstar[k] = g.numpy()

    sigma2_modes = estimate_mode_noise(grads_at_wstar, eigvecs)
    # KEY FIX: rescale noise variance from B_est to target batch size B.
    # Sigma_0(B) = Sigma_single / B, so multiplying the per-mode variances
    # by (B_est / B) gives the correct noise level for the simulated B.
    sigma2_modes *= (B_est / B)
    print(f"[LogReg] sigma2_modes (rescaled to B={B}): "
          f"min={sigma2_modes.min():.3e}, max={sigma2_modes.max():.3e}, "
          f"trace={sigma2_modes.sum():.3e}")

    results = []

    K = 2000
    n_traj = 200

    for s in [0.5, 1.5, 2.5]:
        eta = s / lam_max
        for beta in [0.0, 0.2, 0.4, 0.6]:
            if s >= 2.0 * (1.0 + beta) - 1e-9:
                continue

            # ---- RATE: deterministic full-batch GD from w* + perturbation ----
            rng_rate = np.random.default_rng(999)
            delta = torch.tensor(rng_rate.normal(0, 1e-2, d))
            w_t = w_star + delta
            w_prev = w_t.clone()
            errs = [delta.norm().item()]
            for k in range(100):
                z = -y * (X @ w_t)
                grad = (X.T @ (-y * torch.sigmoid(z))) / N + gamma * w_t
                w_new = w_t - eta * grad + beta * (w_t - w_prev)
                w_prev = w_t
                w_t = w_new
                errs.append((w_t - w_star).norm().item())
            errs = np.array(errs)

            min_idx = int(np.argmin(errs))
            K_fit = min(40, min_idx) if min_idx >= 5 else min(20, len(errs) - 1)
            if K_fit < 5:
                rho_emp = 1.0
            else:
                idx = np.arange(K_fit)
                log_e = np.log(errs[idx])
                slope = np.polyfit(idx, log_e, 1)[0]
                rho_emp = float(np.exp(slope))

            # ---- VARIANCE: stochastic mini-batch SGD at steady state ----
            variances = []
            for traj in range(n_traj):
                rng_t = np.random.default_rng(
                    1000 * int(s * 10) + int(beta * 10) + traj)
                w_t = w_star.clone()
                w_prev = w_t.clone()
                for k in range(K):
                    idx_b = rng_t.integers(0, N, B)
                    Xb = X[idx_b]
                    yb = y[idx_b]
                    z = -yb * (Xb @ w_t)
                    grad = (Xb.T @ (-yb * torch.sigmoid(z))) / B + gamma * w_t
                    w_new = w_t - eta * grad + beta * (w_t - w_prev)
                    w_prev = w_t
                    w_t = w_new
                    if k >= K - 500:
                        variances.append((w_t - w_star).norm().item() ** 2)
            V_emp = float(np.mean(variances))

            # ---- Scalar prediction of the steady-state variance ----
            V_theory = compute_V_theory(eigvals, sigma2_modes, eta, beta)

            results.append((s, beta, rho_emp, V_emp, V_theory))

    # ---- Save CSV ----
    with open("table_logistic.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["s", "beta", "rho_emp", "rho_theory", "V_emp", "V_theory"])
        for s, beta, rho_emp, V_emp, V_theory in results:
            rho_theory = rho_s(s, beta)
            writer.writerow([f"{s:.4f}", f"{beta:.4f}",
                             f"{rho_emp:.4f}", f"{rho_theory:.4f}",
                             f"{V_emp:.4f}", f"{V_theory:.4f}"])

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=(8, 5))
    for s in [0.5, 1.5, 2.5]:
        sub = [(b, r, v) for (ss, b, r, _, v) in results if ss == s]
        if not sub:
            continue
        rho_med = [x[1] for x in sub]
        var = [x[2] for x in sub]
        ax.plot(rho_med, var, "o-", label=f"$s={s}$")
    ax.set_xlabel("Empirical rate $\\hat\\rho$")
    ax.set_ylabel("Empirical variance $\\hat V$")
    ax.set_title("Logistic regression: empirical rate–variance curves")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("fig5_logistic.png")
    plt.close()
    print("Saved fig5_logistic.png and table_logistic.csv")

# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
if __name__ == "__main__":
    print("=== Running all experiments ===")
    verify_formulas()
    fig1()
    fig2()
    fig3()
    fig4()
    fig6()
    fig7()
    try:
        logistic_experiment()
    except ImportError:
        print("[LogReg] PyTorch not installed; skipping.")
    print("=== Done ===")

=== Running all experiments ===

=== Formula verification against Lyapunov solve ===
  HB   s=0.50 beta=0.30: Lyapunov=0.442177, formula=0.442177, rel_err=1.26e-16
  HB   s=1.50 beta=0.40: Lyapunov=1.196581, formula=1.196581, rel_err=0.00e+00
  Nest s=0.50 beta=0.30: Lyapunov=0.375817, formula=0.375817, rel_err=2.95e-16
  Nest s=0.80 beta=0.50: Lyapunov=0.698413, formula=0.698413, rel_err=0.00e+00
Saved fig1_det.png
[Fig2] Max relative discrepancy: 0.321%
Saved fig3_hb_rho.png


/tmp/ipykernel_16/2788468460.py:69: RuntimeWarning: divide by zero encountered in scalar divide
  return s * (1.0 + beta) / ((1.0 - beta) * (2.0 * (1.0 + beta) - s))


Saved fig4_pareto.png
[Fig6] Exact switch beta* = (1-sqrt(0.1))^2 = 0.4675
[Fig7] s=1.5: Nesterov stable for beta in (0.000, 0.500)
[Fig7] s=1.8: Nesterov stable for beta in (0.000, 0.125)
Saved fig7_nesterov.png
[LogReg] lam_max=0.3213, lam_min=0.2005, kappa=1.603
[LogReg] sigma2_modes (rescaled to B=50): min=7.084e-05, max=2.308e-03, trace=3.575e-02
Saved fig5_logistic.png and table_logistic.csv
=== Done ===
